In [ ]:
# 確認有拿到免費 GPU（執行前記得：上方選單 "執行階段" → "變更執行階段類型" → 選 GPU）
import torch
print(f"GPU 可用: {torch.cuda.is_available()}")
print(f"裝置: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

!pip install transformers datasets -q

GPU 可用: True
裝置: Tesla T4


In [ ]:
from datasets import load_dataset

dataset = load_dataset("lansinuote/ChnSentiCorp")

train_texts = list(dataset['train']['text'])
train_labels = list(dataset['train']['label'])  # 0=negative, 1=positive

val_texts = list(dataset['validation']['text'])
val_labels = list(dataset['validation']['label'])

print(f"訓練資料: {len(train_texts)} 筆")
print(f"驗證資料: {len(val_texts)} 筆")
print(f"範例: {train_texts[0]} → label: {train_labels[0]}")

訓練資料: 9600 筆
驗證資料: 1200 筆
範例: 选择珠江花园的原因就是方便，有电动扶梯直接到达海边，周围餐馆、食廊、商场、超市、摊位一应俱全。酒店装修一般，但还算整洁。 泳池在大堂的屋顶，因此很小，不过女儿倒是喜欢。 包的早餐是西式的，还算丰富。 服务吗，一般 → label: 1


In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import Dataset
import torch

model_name = "hfl/chinese-macbert-base"
tokenizer = BertTokenizer.from_pretrained(model_name)

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        item = {k: v.squeeze(0) for k, v in encoding.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = SentimentDataset(train_texts, train_labels, tokenizer)
val_dataset = SentimentDataset(val_texts, val_labels, tokenizer)

print(f"訓練集大小: {len(train_dataset)}")

訓練集大小: 9600


In [ ]:
!pip install -U transformers -q

model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2)

from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='weighted')
    return {'accuracy': acc, 'f1': f1}

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=200,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 72.0 MB/s eta 0:00:00


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: hfl/chinese-macbert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params 

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.272690,0.249323,0.917500,0.917488
2,0.157409,0.236182,0.935833,0.935784
3,0.075346,0.270292,0.945833,0.945828


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1800, training_loss=0.19582717696825663, metrics={'train_runtime': 736.6031, 'train_samples_per_second': 39.098, 'train_steps_per_second': 2.444, 'total_flos': 1894399598592000.0, 'train_loss': 0.19582717696825663, 'epoch': 3.0})

In [ ]:
model.save_pretrained("./my_finetuned_bert")
tokenizer.save_pretrained("./my_finetuned_bert")

# 建議掛載 Google Drive，把模型存起來避免 Colab 斷線後消失
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copytree("./my_finetuned_bert", "/content/drive/MyDrive/my_finetuned_bert", dirs_exist_ok=True)
print("模型已存到 Google Drive")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
模型已存到 Google Drive


In [ ]:
from google.colab import files
uploaded = files.upload()  # 這裡選擇上傳你的 to_label.csv

import pandas as pd
test_df = pd.read_csv("to_label.csv")  # 檔名依你上傳的檔案調整
print(test_df.head())

Saving to_label.csv to to_label (1).csv
                  date                                             title  \
0  2024-02-20 05:15:00           全球前25強半導體公司出爐！台積電、聯電皆上榜冠軍是它 - Yahoo奇摩新聞   
1  2024-07-04 05:50:13  台股再刷歷史新高！勁揚350點收23522點台積電躋身第19千金股| 聯合新聞網 - 聯合新聞網   
2  2024-06-19 06:45:03    台股收盤／站穩2萬3！台積電狂漲38元飆981元 指數收23209點再創新高 - Yahoo   
3  2024-07-04 05:50:13       台股再刷歷史新高！勁揚350點收23,522點 台積電晉身第19千金股 - 聯合新聞網   
4  2024-03-12 23:12:00                    台積電旺、台灣主要IT廠營收減幅縮小 - Yahoo奇摩股市   

   label  
0      1  
1      1  
2      1  
3      1  
4      1  


In [ ]:
from torch.nn.functional import softmax

model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def predict_sentiment(text):
    inputs = tokenizer(str(text), truncation=True, padding='max_length',
                        max_length=128, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = softmax(outputs.logits, dim=1)
    pred = torch.argmax(probs, dim=1).item()
    confidence = probs[0][pred].item()
    # 0=negative, 1=positive；因為訓練資料沒有中立類別
    # 用信心分數判斷：如果最高機率 < 0.6，視為「不確定/中立」
    if confidence < 0.6:
        return 0, confidence  # 中立
    else:
        return (1 if pred == 1 else -1), confidence

test_df['pred'], test_df['confidence'] = zip(*test_df['title'].apply(predict_sentiment))

from sklearn.metrics import classification_report
print(classification_report(test_df['label'], test_df['pred']))

              precision    recall  f1-score   support

          -1       0.45      0.93      0.60        28
           0       0.40      0.08      0.13        25
           1       0.84      0.75      0.79        97

    accuracy                           0.67       150
   macro avg       0.56      0.59      0.51       150
weighted avg       0.69      0.67      0.65       150

